### 1.1 Load data

In [1]:
from datasets import load_dataset

SEED = 640
SLICE = 640
N_SAMPLES = 5000

dataset = load_dataset(
    "cardiffnlp/tweet_eval",
    "sentiment",
    split="train"
)

end = min(SLICE + N_SAMPLES, len(dataset))
subset = dataset.select(range(SLICE, end))

print("Dataset size:", len(subset))
print("First example:", subset[0])

README.md:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

sentiment/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 3.78MB            

sentiment/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

sentiment/test-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B /  901kB            

sentiment/test-00000-of-00001.parquet: downloading bytes:           |  0.00B            

...

sentiment/validation-00000-of-00001.parq(…): downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/45615 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/12284 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2000 [00:00<?, ? examples/s]

Dataset size: 5000
First example: {'text': 'Interesting Sunday read on the rise, fall and rebirth of Google+.', 'label': 2}


### 1.2 Split data

In [2]:
from sklearn.model_selection import train_test_split
from collections import Counter
import json
from pathlib import Path

# Indices within our fixed 5,000-row subset
indices = list(range(len(subset)))
labels = subset["label"]

# 70% train, 30% temporary
train_idx, temp_idx = train_test_split(
    indices,
    test_size=0.30,
    random_state=SEED,
    stratify=labels
)

# Split the remaining 30% equally
val_idx, test_idx = train_test_split(
    temp_idx,
    test_size=0.50,
    random_state=SEED,
    stratify=[labels[i] for i in temp_idx]
)

train_dataset = subset.select(train_idx)
val_dataset = subset.select(val_idx)
test_dataset = subset.select(test_idx)

label_names = ["Negative", "Neutral", "Positive"]

for name, ds in [
    ("Train", train_dataset),
    ("Validation", val_dataset),
    ("Test", test_dataset)
]:
    counts = Counter(ds["label"])
    print(f"{name}: {len(ds)} samples")
    print({
        label_names[i]: counts.get(i, 0)
        for i in range(3)
    })

# Save the split for HW4.5
output_dir = Path("hw4a_results")
output_dir.mkdir(exist_ok=True)

split_info = {
    "dataset": "cardiffnlp/tweet_eval",
    "subset": "sentiment",
    "source_split": "train",
    "seed": SEED,
    "slice": SLICE,
    "row_start": SLICE,
    "row_end_inclusive": end - 1,
    "train_indices": train_idx,
    "val_indices": val_idx,
    "test_indices": test_idx
}

with open(output_dir / "split_indices.json", "w") as f:
    json.dump(split_info, f, indent=2)

print("\nSplit indices saved successfully!")

Train: 3500 samples
{'Negative': 543, 'Neutral': 1602, 'Positive': 1355}
Validation: 750 samples
{'Negative': 116, 'Neutral': 344, 'Positive': 290}
Test: 750 samples
{'Negative': 116, 'Neutral': 343, 'Positive': 291}

Split indices saved successfully!


### 1.3 BERT Tokenization\

In [4]:
import torch
from torch.utils.data import TensorDataset
from transformers import BertTokenizer

MODEL_NAME = "bert-base-uncased"
MAX_LENGTH = 128

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME)

def tokenize_dataset(dataset):
    # Convert Hugging Face Column to Python list
    texts = list(dataset["text"])
    labels = list(dataset["label"])

    encodings = tokenizer(
        texts,
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt"
    )

    labels = torch.tensor(labels, dtype=torch.long)

    return TensorDataset(
        encodings["input_ids"],
        encodings["attention_mask"],
        labels
    )

train_tensor_dataset = tokenize_dataset(train_dataset)
val_tensor_dataset = tokenize_dataset(val_dataset)
test_tensor_dataset = tokenize_dataset(test_dataset)

print("Train samples:", len(train_tensor_dataset))
print("Validation samples:", len(val_tensor_dataset))
print("Test samples:", len(test_tensor_dataset))

# Inspect one sample
input_ids, attention_mask, label = train_tensor_dataset[0]

print("\nInput IDs shape:", input_ids.shape)
print("Attention mask shape:", attention_mask.shape)
print("Label:", label.item())

print("\nDecoded text:")
print(tokenizer.decode(input_ids, skip_special_tokens=True))

Train samples: 3500
Validation samples: 750
Test samples: 750

Input IDs shape: torch.Size([128])
Attention mask shape: torch.Size([128])
Label: 0

Decoded text:
" estimated 5 % of muslims in britain r islamists, not many u may think, but there r about 6, 000, 000 muslims in britain. that ' s 300, 000 islamists "


### 1.4 DataLoader & BERT Model Initialization

In [5]:
import torch
import numpy as np
import random
from torch.utils.data import DataLoader

# Reproducibility
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

# Device
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Device:", device)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU Memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory
            / 1024**3, 2
        ),
        "GB"
    )

# Training configuration
BATCH_SIZE = 16
EPOCHS = 3
LEARNING_RATE = 2e-5

train_generator = torch.Generator()
train_generator.manual_seed(SEED)

train_loader = DataLoader(
    train_tensor_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    generator=train_generator
)

val_loader = DataLoader(
    val_tensor_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_tensor_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

print("\nTrain batches:", len(train_loader))
print("Validation batches:", len(val_loader))
print("Test batches:", len(test_loader))

Device: cpu

Train batches: 219
Validation batches: 47
Test batches: 47


In [6]:
from transformers import BertForSequenceClassification
from torch.optim import AdamW

NUM_LABELS = 3

id2label = {
    0: "Negative",
    1: "Neutral",
    2: "Positive"
}

label2id = {
    name: idx for idx, name in id2label.items()
}

model = BertForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2label,
    label2id=label2id
)

model = model.to(device)

optimizer = AdamW(
    model.parameters(),
    lr=LEARNING_RATE,
    weight_decay=0.01
)

total_params = sum(
    p.numel() for p in model.parameters()
)

trainable_params = sum(
    p.numel() for p in model.parameters()
    if p.requires_grad
)

print("Model:", MODEL_NAME)
print("Number of classes:", NUM_LABELS)
print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Optimizer: AdamW")
print("Learning rate:", LEARNING_RATE)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  440MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Model: bert-base-uncased
Number of classes: 3
Total parameters: 109484547
Trainable parameters: 109484547
Optimizer: AdamW
Learning rate: 2e-05


### 1.5 BERT Fine-Tuning — Training Loop

In [ ]:
import time
import json
import pandas as pd
from pathlib import Path

OUTPUT_DIR = Path("hw4a_results")
BEST_MODEL_DIR = OUTPUT_DIR / "best_bert_model"
OUTPUT_DIR.mkdir(exist_ok=True)

def evaluate(model, data_loader):
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total_samples = 0

    with torch.inference_mode():
        for input_ids, attention_mask, labels in data_loader:
            input_ids = input_ids.to(device)
            attention_mask = attention_mask.to(device)
            labels = labels.to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                labels=labels
            )

            batch_size = labels.size(0)

            total_loss += outputs.loss.item() * batch_size
            predictions = outputs.logits.argmax(dim=1)

            total_correct += (
                predictions == labels
            ).sum().item()

            total_samples += batch_size

    avg_loss = total_loss / total_samples
    accuracy = total_correct / total_samples

    return avg_loss, accuracy


history = []
best_val_loss = float("inf")
best_epoch = None

if device.type == "cuda":
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()

start_time = time.perf_counter()

for epoch in range(EPOCHS):
    model.train()

    total_train_loss = 0.0
    total_train_samples = 0

    if device.type == "cuda":
        torch.cuda.synchronize()

    train_start = time.perf_counter()

    for input_ids, attention_mask, labels in train_loader:
        input_ids = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        labels = labels.to(device)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        batch_size = labels.size(0)
        total_train_loss += loss.item() * batch_size
        total_train_samples += batch_size

    if device.type == "cuda":
        torch.cuda.synchronize()

    train_duration = time.perf_counter() - train_start
    train_loss = total_train_loss / total_train_samples

    val_loss, val_accuracy = evaluate(model, val_loader)

    # Save the checkpoint with the lowest validation loss
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        best_epoch = epoch + 1
        model.save_pretrained(BEST_MODEL_DIR)
        tokenizer.save_pretrained(BEST_MODEL_DIR)

    history.append({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "val_accuracy": val_accuracy,
        "train_time_sec": train_duration,
        "train_samples_per_sec": total_train_samples / train_duration
    })

    print(
        f"Epoch [{epoch+1}/{EPOCHS}] "
        f"Train Loss: {train_loss:.4f} | "
        f"Val Loss: {val_loss:.4f} | "
        f"Val Acc: {val_accuracy:.4f} | "
        f"Train Time: {train_duration:.2f}s"
    )

if device.type == "cuda":
    torch.cuda.synchronize()

total_training_time = time.perf_counter() - start_time

peak_gpu_memory_gb = (
    torch.cuda.max_memory_allocated() / (1024 ** 3)
    if device.type == "cuda" else None
)

history_df = pd.DataFrame(history)
history_df.to_csv(
    OUTPUT_DIR / "training_history.csv",
    index=False
)

print("\nTraining completed!")
print(f"Best validation-loss epoch: {best_epoch}")
print(f"Best validation loss: {best_val_loss:.4f}")
print(f"Total wall-clock time: {total_training_time:.2f}s")
print(f"Peak GPU allocated memory: {peak_gpu_memory_gb} GB")

baseline_metrics = {
    "model": MODEL_NAME,
    "seed": SEED,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "max_length": MAX_LENGTH,
    "precision": "FP32",
    "best_epoch_by_val_loss": best_epoch,
    "best_val_loss": best_val_loss,
    "total_training_time_sec": total_training_time,
    "peak_gpu_allocated_memory_gb": peak_gpu_memory_gb,
    "gpu": (
        torch.cuda.get_device_name(0)
        if device.type == "cuda" else str(device)
    )
}

with open(OUTPUT_DIR / "baseline_metrics.json", "w") as f:
    json.dump(baseline_metrics, f, indent=2)